In [ ]:
import numpy as np
import pandas as pd
import os

INPUT_DIR = os.environ.get('INPUT_DIR', 'data')
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', 'output')

os.makedirs(OUTPUT_DIR, exist_ok=True)

trans_df = pd.read_csv(f"{INPUT_DIR}/LI-Small_Trans.csv")
trans_df.head(5)

In [ ]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df['Timestamp'].min()},{trans_df['Timestamp'].max()}]")

In [ ]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Series([], Name: Bank, dtype: int64)

In [ ]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 184033


In [ ]:
# Analyze accounts.
accounts_df = pd.read_csv(f"{INPUT_DIR}/LI-Small_accounts.csv")
print("SIZE:", accounts_df.shape[0])
accounts_df.head(5)

In [ ]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 100497


In [ ]:
def filter_function(x):
    unique_account_size = x.groupby(["To Bank", "Account.1"]).size().size
    return  unique_account_size > 5 and unique_account_size < 10
ranged_trans_usd_sept_df = trans_usd_sept_1st_df.groupby(["From Bank", "Account"]).filter(filter_function)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 1459


In [ ]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']]
low_profile_transactions.to_csv(f"{OUTPUT_DIR}/query_1.csv", index=False)
low_profile_transactions

In [ ]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
result_q2 = max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]]
result_q2.to_csv(f"{OUTPUT_DIR}/query_2.csv", index=False)
result_q2

In [ ]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})
lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
result_q3 = lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]]
result_q3.to_csv(f"{OUTPUT_DIR}/query_3.csv", index=False)
result_q3

In [ ]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

accounts_df = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
account_pairs_df = accounts_df.merge(accounts_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] > 5)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts.to_csv(f"{OUTPUT_DIR}/query_4.csv", index=False)
unique_accounts

In [ ]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
#Map everything to US Dollars by a fixed table.
TO_US_DOLLARS = {
    "Australian Dollar": 0.72,
    "Bitcoin": 78.33,
    "Brazil Real": 0.20,
    "Canadian Dollar": 0.73,
    "Euro": 1.17,
    "Mexican Peso": 0.06,
    "Ruble": 0.01,
    "Rupee": 0.01,
    "Saudi Riyal":0.27,
    "Shekel": 0.33,
    "Swiss Franc": 1.27,
    "UK Pound":  1.35,
    "US Dollar": 1,
    "Yen": 0.006,
    "Yuan": 0.15
}
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df['Amount Paid'] * trans_sept_1st_wire_or_ach_converted_df['Payment Currency'].map(TO_US_DOLLARS)
result_q5 = trans_sept_1st_wire_or_ach_converted_df[trans_sept_1st_wire_or_ach_converted_df['Amount'] < 1]
result_q5.to_csv(f"{OUTPUT_DIR}/query_5.csv", index=False)
print("SIZE:", result_q5.shape[0])